In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Download dataset from Kaggle
Q1_data = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(Q1_data)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('delivery_time Distribution')
plt.xlabel('delivery_time')
plt.ylabel('Time')
plt.show()

In [ ]:
# Task 1 & 2:
df_cleaned = df.drop("Order_ID", axis=1)

print("Missing values before imputation:")
print(df_cleaned.isnull().sum()[df_cleaned.isnull().sum() > 0])

for col in df_cleaned.select_dtypes(include=np.number).columns:
    if df_cleaned[col].isnull().any():
        median_val = df_cleaned[col].median()
        df_cleaned[col].fillna(median_val, inplace=True)


for col in df_cleaned.select_dtypes(include='object').columns:
    if df_cleaned[col].isnull().any():
        mode_val = df_cleaned[col].mode()[0]
        df_cleaned[col].fillna(mode_val, inplace=True)

print("\nMissing values after imputation:")
print(df_cleaned.isnull().sum()[df_cleaned.isnull().sum() > 0])

df = df_cleaned.copy()

In [ ]:
# Task 3:
def check_duplicates(df_param):
  duplicates = df_param.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_param.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")
  return df_param

df = check_duplicates(df)

In [ ]:
# Task 4:
categorical_cols = df.select_dtypes(include='object').columns.tolist()

if categorical_cols:
    print(f"Categorical Columns to encode: {categorical_cols}")
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    encoded_features = encoder.fit_transform(df[categorical_cols])
    encoded_df = pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out(categorical_cols), index=df.index)

    df = pd.concat([df.drop(columns=categorical_cols), encoded_df], axis=1)
    print("\nDataFrame after One-Hot Encoding:")
    display(df.head())
else:
    print("No categorical columns found for encoding.")

In [ ]:
# Task 5:
X_features = df.drop(columns=['Delivery_Time'])
y_target = df['Delivery_Time']


numerical_cols_for_scaling = X_features.select_dtypes(include=np.number).columns.tolist()

if numerical_cols_for_scaling:
    print(f"Applying StandardScaler to {len(numerical_cols_for_scaling)} numerical features.")
    scaler = StandardScaler()
    X_features[numerical_cols_for_scaling] = scaler.fit_transform(X_features[numerical_cols_for_scaling])
    print("\nFeatures after scaling:")
    display(X_features.head())
else:
    print("No numerical features found for scaling.")


df = pd.concat([X_features, y_target], axis=1)

In [ ]:
# # Task 5: Write your code here:

features = df.columns.drop("Delivery_Time")   # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])
df.head()

In [ ]:
# Task 1:
X = df.drop(columns=['Delivery_Time'])
y = df['Delivery_Time']

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

In [ ]:
# Task 2,3,4,5:
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

mae_scores = []
models = []

print(f"Performing {n_splits}-fold cross-validation...")

for fold, (train_index, test_index) in enumerate(kf.split(X, y)):
    print(f"\n--- Fold {fold+1}/{n_splits} ---")
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]


    model = RandomForestRegressor(random_state=42)
    model.fit(X_train, y_train)
    models.append(model)


    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)
    print(f"Mean Absolute Error (MAE) for Fold {fold+1}: {mae:.4f}")


print(f"\nAverage Mean Absolute Error (MAE) across {n_splits} folds: {np.mean(mae_scores):.4f}")
print(f"Standard Deviation of MAE across {n_splits} folds: {np.std(mae_scores):.4f}")


best_model_idx = np.argmin(mae_scores)
trained_model = models[best_model_idx]

In [ ]:
# Task 1:
import seaborn as sns

if 'trained_model' in locals():
    feature_importances = trained_model.feature_importances_
    features = X.columns
    importance_df = pd.DataFrame({'Feature': features, 'Importance': feature_importances})
    importance_df = importance_df.sort_values(by='Importance', ascending=False)

    plt.figure(figsize=(12, 7))
    sns.barplot(x='Importance', y='Feature', data=importance_df)
    plt.title('Feature Importances from RandomForestRegressor')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()
else:
    print("No trained model found. Please run the modeling cell first.")

In [ ]:
# Task 2:

full_model = RandomForestRegressor(random_state=42)
full_model.fit(X, y)
y_pred_full = full_model.predict(X)

plt.figure(figsize=(10, 5))
plt.hist(y_pred_full, bins=30, edgecolor='black', alpha=0.7, label='Predicted Delivery Time')
plt.hist(y, bins=30, edgecolor='red', alpha=0.5, label='Actual Delivery Time') # Overlay actual for comparison
plt.title('Distribution of Predicted vs Actual Delivery Time')
plt.xlabel('Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.legend()
plt.show()

In [ ]:



n_splits_ensemble = 5
kf_ensemble = KFold(n_splits=n_splits_ensemble, shuffle=True, random_state=42)

ensemble_mae_scores = []

print(f"Performing {n_splits_ensemble}-fold cross-validation with ensemble models...")

for fold, (train_index, test_index) in enumerate(kf_ensemble.split(X, y)):
    print(f"\n--- Ensemble Fold {fold+1}/{n_splits_ensemble} ---")
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]


    rf_model = RandomForestRegressor(random_state=42)
    rf_model.fit(X_train, y_train)
    rf_preds = rf_model.predict(X_test)

    rf_model = RandomForestClassifier(random_state=42)
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_test)



    ensemble_preds = (rf_preds + rf_pred) / 2


    mae = mean_absolute_error(y_test, ensemble_preds)
    ensemble_mae_scores.append(mae)
    print(f"Ensemble Mean Absolute Error (MAE) for Fold {fold+1}: {mae:.4f}")

print(f"\nAverage Ensemble Mean Absolute Error (MAE) across {n_splits_ensemble} folds: {np.mean(ensemble_mae_scores):.4f}")
print(f"Standard Deviation of Ensemble MAE across {n_splits_ensemble} folds: {np.std(ensemble_mae_scores):.4f}")